# Knowledge Representation
## Text Analysis, Take 2
Garth Mortensen  
2019.09.19

## Overview

This analysis builds upon cleaning.ipynb, by shifting from [Project Gutenberg](http://www.gutenberg.org/) to [Pubmed](https://www.ncbi.nlm.nih.gov/pubmed/) as a source for text. In doing so, I now employ beautifulsoup library to target text appearing in the Abstract element.

## Read Text

Define two conditions of interest, and construct top 20 URLs.

In [ ]:
from urllib.request import urlopen
from bs4 import BeautifulSoup

conditions = ["diabetes", "cancer"]

condition_urls = []


def construction_url(condition_list):

    for each_condition_list in condition_list:
        
        journal = "%5BJOUR%5D"  # 5D for most recent, I think.
        condition_url = "https://www.ncbi.nlm.nih.gov/pubmed/?term=" + each_condition_list + journal
        condition_urls.append(condition_url)

    return condition_urls


construction_url_output = construction_url(conditions)
print("INITIAL:", construction_url_output)

Get top 20 articles

In [ ]:
listy = [1, 4, 2, 4, 6, 7, 4, 2, 3]

  
# printing modified list 
print(listy) 

In [ ]:
def get_20_articles(get_20_articles_input):

    html = urlopen(get_20_articles_input).read()

    # parse the html using beautiful soup
    soup = BeautifulSoup(html, 'html.parser')

    all_articles = []
    url_start = "https://www.ncbi.nlm.nih.gov/pubmed/"

    articles = soup.find_all(attrs={'class': 'title'})
    # print(articles)

    # process main, starter page, and extract list of urls
    for article in articles:

        # split by uid, which is article id
        link = str(article).split('uid=')
        # print(link[1])

        # remove anything after the uid, and define the number
        uid = str(link[1]).split('&amp')[0]
        # print(uid)

        # build the complete url
        url = url_start + uid

        all_articles.append(url)

    return all_articles


get_20_articles_output_0 = get_20_articles(construction_url_output[0])
get_20_articles_output_1 = get_20_articles(construction_url_output[1])

unwanted_0 = [7]
for ele in sorted(unwanted_0, reverse = True):
    del get_20_articles_output_0[ele]

unwanted_1 = [0, 1, 8, 13, 19]
for ele in sorted(unwanted_1, reverse = True):
    del get_20_articles_output_1[ele]

In [ ]:
url_start = "https://www.ncbi.nlm.nih.gov/pubmed/"
url_list = ["31582408","31578193","31578192","31540941","31540940","31540938","31420409","31399432","31391173","31391172","31375510","31371518","31350303","31345937","31337616","31311801","31311800","31292135","31292134","31537525","31537524","31530579","31519700","31519699","31506345","31506344","31506343","31492662","31492661","31431441","31431440","31217176","31217175","31217174","31201281","31201280","31175102","31175101","31175100","31171562","31167879","31167878","31167877","30862679","31471292","31451517","31439648","31439647","31439646","31439645","31439644","31439643","31439642","31439641","31399433","31399431","31331989","31331988","31331987","31178433","31167880","31127058","31127057","31127056","31127055","31127053","31127052","31092480","31092478","31088856","31088855","31088854","31221802","31221801","31221800","31127054","31092479","31092477","31048370","31048369","31048368","31048367","31010960","31010959","31010958","31010957","31010956","31010955","30962220","30962219","31178432","31109941","31109940","31109939","31109938","30967424","30936150","30936149","30936148","30936147","30936146","30936145","30936144","30936143","30936142","30936140","30894367","30894366","30885990","30885989","30862683","30862682","30862681","30862680","31010883","31010882","31010881","31010880","31010879","31010878","31010877","30936151","30936141","30862678","30833470","30833469","30833468","30833467","30833466","30833465","30796029","30796028","30765337","30765336","30765335","30733330","30728185","30626611","30455375","30894392","30755400","30745441","30728184","30728183","30696708","30692245","30679185","30679184","30679183","30674623","30674622","30670477","30655386","30655385","30626610","30626607","30617218","30787070","30787069","30787068","30787067","30787066","30635275","30635274","30626609","30626608","30617219","30552111","30552110","30552109","30552108","30552107","30523028","30523027","30523026","30523025","30523024","30530781","30409780","30305367","31069088","30796027","30816853","30816852","30665957","30665956","30665955","30665954","30665953","30665952","30665951","30552106","30487265","30487264","30487263","30487262","30455377","30455376","30425064","30425063"]

all_urls = []

for url in url_list:
    entire_url = url_start + url
    all_urls.append(entire_url)

print(all_urls[0:3])

In [ ]:
def parseURL(url_list):
    """Pull Abstract from PubMed article."""

    all_text = []
    
    for each_url in url_list:
    
        html = urlopen(each_url).read()

        # parse the html using beautiful soup
        soup = BeautifulSoup(html, 'html.parser')

        # Point to the element of interest on pubmed
        name_box = soup.find(attrs={'class': 'abstr'})
        text = name_box.text.strip()

        # remove Abstract from title
        text = text[8:]

        # remove everything after.
        split_text = "Copyright"

        # split on beginning of unwanted text, and take first half
        text = text.split(split_text)[0]

        # remove everything after.
        split_text = "?"

        # remove after copyright symbole of ? American Diabetes Association
        text = text.split(split_text)[0]

        all_text.append(text)
    
    return all_text


parseURL_output_0 = parseURL(get_20_articles_output_0)
parseURL_output_1 = parseURL(get_20_articles_output_1)

print(type(parseURL_output_0))

n = 0

for each_parseURL_output in parseURL_output_0:
    n += 1
    print(str(n) + ":", len(each_parseURL_output))
    print(each_parseURL_output)

## Text Cleaning

Because the text being analyzed in this notebook are pubmed health studies, they are technical langauge. Many numbers, statistics symbols and health-space vocabulary words are used. We should be mindful of what text cleaning operations are applied.

Do various things such as remove punctuation, remove whitespace, etc.

One goal during text cleanup is to remove total word space, so that our resultant vectors are smaller. Think regularization.

In [ ]:
def clean_text(articles_content):
    """Remove unwanted characters, etc from articles"""
    
    articles_content_stripped = []
    
    for each_articles_content in articles_content:

        # lower the article
        each_articles_content = each_articles_content.lower()
        
        # split by white spaces
        words = each_articles_content.split()

        # remove all tokens that are not alphabetic
        words = [word for word in words if word.isalpha()]

        # remove words < char len 2 and > 15
        words = [word for word in words if len(word) > 2 and len(word) < 15]
        
        articles_content_stripped.append(words)
        
    return articles_content_stripped


clean_text_output_0 = clean_text(parseURL_output_0)
clean_text_output_1 = clean_text(parseURL_output_1)

n = 0

for each_clean_text_output in clean_text_output_0:
    n += 1
    print(str(n) + ":", len(each_clean_text_output))

print(clean_text_output_0[0])

### Remove stopwords

The words we will remove are shown below.

We can easily create our own customized stopword list, so as to remove any common, low-information medical terminology. It is just noise.

In [ ]:
from nltk.corpus import stopwords

stop_words = stopwords.words('english')
print("\ntotal English stopwords:", len(stop_words))
# print(stop_words)


def remove_stopwords(input_clean_text_output):
    """Remove stopwords"""
    
    all_texts_w_stopwords_removed = []
    
    for each_input_clean_text_output in input_clean_text_output:

        keep_words = [w for w in each_input_clean_text_output if not w in stop_words]
   
        all_texts_w_stopwords_removed.append(keep_words)

    return all_texts_w_stopwords_removed


remove_stopwords_output_0 = remove_stopwords(clean_text_output_0)
remove_stopwords_output_1 = remove_stopwords(clean_text_output_1)

n = 0

for each_remove_stopwords_output in remove_stopwords_output_0:
    n += 1
    print(str(n) + ":", len(each_remove_stopwords_output))

print(remove_stopwords_output_0[0])

### Stemming

According to [Introduction to Information Retrieval](https://nlp.stanford.edu/IR-book/html/htmledition/stemming-and-lemmatization-1.html) 
>**Stemming** usually refers to a crude heuristic process that chops off the ends of words in the hope of achieving this goal correctly most of the time, and often includes the removal of derivational affixes. **Lemmatization** usually refers to doing things properly with the use of a vocabulary and morphological analysis of words, normally aiming to remove inflectional endings only and to return the base or dictionary form of a word, which is known as the lemma.

Porter's stemmer is the most well known stemmer. Time and place = Cambridge (Oxford), 1980. There are stemmers for other languages that follow other rules. It removes the common morphological and inflextional endings from English words. Its main usei s as part of a term normalization process that is usually done when setting up Information Retrieval systems.

Word -> Stemmer -> Root form

The goal is to reduce the total vocabulary space by simplifying words into their stems, chopping off the suffix.

_cat_ <- car, cats, cat's, cats'  
I suspect that Old English/strong/irregular verbs tend to be harder to stem.
Do, did, done

Weak/regular/latin verbs should be easier to stem, as the suffix determines their tense.
Yell, yelled, yelling

In [ ]:
from nltk.stem.porter import PorterStemmer

porter = PorterStemmer()


def stemmed_words(stemmed_words_input):
    """stem the words"""
    
    all_texts_stemmed = []
    
    for each_stemmed_words_input in stemmed_words_input:

        stemmed = [porter.stem(word) for word in each_stemmed_words_input]
   
        all_texts_stemmed.append(stemmed)

    return all_texts_stemmed


stemmed_words_output_0 = stemmed_words(remove_stopwords_output_0)
stemmed_words_output_1 = stemmed_words(remove_stopwords_output_1)
print(stemmed_words_output_0[0])

#### Merge doc lists into single lines.

Using stemmed_words_output, the cleaned, tokenized version, join the docs back together.

In [ ]:
def joined_docs(joined_docs_input):
    
    total_list = []
    
    for each_joined_docs_input in joined_docs_input:
        
        each_joined_docs_input = ' '.join(each_joined_docs_input)
        
        total_list.append(each_joined_docs_input)
        
    return total_list


joined_docs_output_0 = joined_docs(stemmed_words_output_0)
joined_docs_output_1 = joined_docs(stemmed_words_output_1)

for each in joined_docs_output_0:
    print("joined_docs_output_0")
    print(each)
    print("\n")

for each in joined_docs_output_1:
    print("joined_docs_output_1")
    print(each)
    print("\n")

So up to this point, I've cleaned the text, and then joined each document into it's own list element. Now, I think I need to
  1. Keras tokenize
  2. ?
  
Stop this monkey work. Just construct a dataset.

70 train, 30 test

train = 35 cancer, 35 diabetes

test = 15 cancer, 15 diabetes


Similarity between documents is calculated as the dot product between the vectors. 

        book my dog

doc1 = [1    2  0]

doc2 = [0    0  5]

doc3 = [2    4  0]

doc1 * doc2 = 0. There is no overlap in vector direction, so there is 0 similarity.

doc2 * doc3 = similarity exists.